# Dimensionless periodic Cahn--Hilliard evolution

## Prerequisites

See the [Cahn--Hilliard lecture note](../../../docs/lecturenotes/solidstate/phase-field/cahn-hilliard/index.md). This laboratory assumes conserved scalar fields, variational chemical potential, periodic Fourier differentiation, and semi-implicit time integration.

## Objectives

1. Evolve deterministic reduced one- and two-dimensional periodic fields.
2. Preserve the zero Fourier mode and therefore the discrete mean field.
3. Retain bounded immutable snapshots including the final step.
4. Visualize the development of diffuse composition structure.

The parameters, coordinates, time, and fields are dimensionless illustrative values. They are not mapped to a particular alloy or experimental system.


## Reduced model and update

For the reduced free energy

$$
F[c]=\int\left[\frac14(c^2-1)^2+\frac{\kappa}{2}|\nabla c|^2\right]d\mathbf x,
$$

the chemical potential and constant-mobility dynamics are

$$
\mu=c^3-c-\kappa\nabla^2c,
\qquad
\frac{\partial c}{\partial t}=M\nabla^2\mu.
$$

On a periodic grid, the maintained solver advances

$$
\widehat c^{n+1}=
\frac{\widehat c^n-\Delta t\,M k^2\widehat{(c^3-c)}^n}
{1+\Delta t\,M\kappa k^4}.
$$

The nonlinear bulk term is explicit and the fourth-order gradient term is implicit. The implementation does not apply spectral de-aliasing or claim unconditional energy stability.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from projectkoios.physkit.solidstate.phase_field.cahn_hilliard.base import (
    DimensionlessCahnHilliardParameters,
)
from projectkoios.physkit.solidstate.phase_field.cahn_hilliard.one_dimensional.model import (  # noqa: E501
    DimensionlessPeriodicCahnHilliard1DModel,
    DimensionlessPeriodicCahnHilliard1DState,
)
from projectkoios.physkit.solidstate.phase_field.cahn_hilliard.two_dimensional.model import (  # noqa: E501
    DimensionlessPeriodicCahnHilliard2DModel,
    DimensionlessPeriodicCahnHilliard2DState,
)
from projectkoios.physkit.units import (
    MatrixQuantity,
    ScalarQuantity,
    Unitless,
    VectorQuantity,
)

In [ ]:
parameters = DimensionlessCahnHilliardParameters(
    mobility=ScalarQuantity(magnitude=1.0, unit=Unitless()),
    gradient_penalty=ScalarQuantity(magnitude=1.0e-2, unit=Unitless()),
    time_step=ScalarQuantity(magnitude=1.0e-5, unit=Unitless()),
)
generator = np.random.default_rng(20260930)

one_dimensional_initial_state = DimensionlessPeriodicCahnHilliard1DState(
    field_values=VectorQuantity(
        magnitude=np.asarray(
            0.02 * generator.standard_normal(256),
            dtype=np.float64,
        ),
        unit=Unitless(),
    ),
    domain_length=ScalarQuantity(magnitude=1.0, unit=Unitless()),
)
one_dimensional_solution = DimensionlessPeriodicCahnHilliard1DModel(
    parameters=parameters,
).solve(
    initial_state=one_dimensional_initial_state,
    step_count=2000,
    snapshot_stride=400,
)

In [ ]:
two_dimensional_initial_state = DimensionlessPeriodicCahnHilliard2DState(
    field_values=MatrixQuantity(
        magnitude=np.asarray(
            0.02 * generator.standard_normal((48, 48)),
            dtype=np.float64,
        ),
        unit=Unitless(),
    ),
    domain_length_x=ScalarQuantity(magnitude=1.0, unit=Unitless()),
    domain_length_y=ScalarQuantity(magnitude=1.0, unit=Unitless()),
)
two_dimensional_solution = DimensionlessPeriodicCahnHilliard2DModel(
    parameters=parameters,
).solve(
    initial_state=two_dimensional_initial_state,
    step_count=1000,
    snapshot_stride=500,
)

In [ ]:
one_dimensional_mean_drift = abs(
    one_dimensional_solution.final_state.mean_field.magnitude
    - one_dimensional_initial_state.mean_field.magnitude
)
two_dimensional_mean_drift = abs(
    two_dimensional_solution.final_state.mean_field.magnitude
    - two_dimensional_initial_state.mean_field.magnitude
)
assert one_dimensional_mean_drift < 1.0e-14
assert two_dimensional_mean_drift < 1.0e-14
assert np.all(np.isfinite(one_dimensional_solution.final_state.field_values.magnitude))
assert np.all(np.isfinite(two_dimensional_solution.final_state.field_values.magnitude))
assert one_dimensional_solution.step_indices[-1] == 2000
assert two_dimensional_solution.step_indices[-1] == 1000
one_dimensional_mean_drift, two_dimensional_mean_drift

In [ ]:
figure, (line_axis, initial_axis, final_axis) = plt.subplots(
    nrows=1,
    ncols=3,
    figsize=(15.0, 4.5),
)
for snapshot_index, snapshot in zip(
    one_dimensional_solution.step_indices,
    one_dimensional_solution.snapshots,
    strict=True,
):
    line_axis.plot(
        snapshot.grid_points.magnitude,
        snapshot.field_values.magnitude,
        label=f"step {snapshot_index}",
    )
line_axis.set(
    xlabel="reduced coordinate",
    ylabel="reduced conserved field",
    title="One-dimensional snapshots",
)
line_axis.grid(True)
line_axis.legend(fontsize="small")

color_limit = max(
    float(np.max(np.abs(two_dimensional_initial_state.field_values.magnitude))),
    float(np.max(np.abs(two_dimensional_solution.final_state.field_values.magnitude))),
)
initial_image = initial_axis.imshow(
    two_dimensional_initial_state.field_values.magnitude,
    origin="lower",
    cmap="RdBu_r",
    vmin=-color_limit,
    vmax=color_limit,
)
initial_axis.set(title="Two-dimensional initial field")
final_axis.imshow(
    two_dimensional_solution.final_state.field_values.magnitude,
    origin="lower",
    cmap="RdBu_r",
    vmin=-color_limit,
    vmax=color_limit,
)
final_axis.set(title="Two-dimensional final field")
figure.colorbar(
    initial_image,
    ax=(initial_axis, final_axis),
    label="reduced conserved field",
    shrink=0.8,
)
figure.subplots_adjust(wspace=0.32)
plt.show()

## Interpretation

Both calculations retain their initial discrete means to floating-point precision because the update leaves the zero Fourier mode unchanged. The displayed structures are calculated outputs of the stated reduced numerical model, not observations of a physical material.

The original exploratory notebook also included both one- and two-dimensional cells. They now share one explicit parameter contract while retaining dimension-specific state and solver ownership.

## Exercises

1. Change the initial mean while preserving the random perturbation amplitude.
2. Compare snapshot behavior for smaller time steps at fixed final reduced time.
3. Increase spatial resolution and examine whether the represented structures are grid-sensitive.
4. Add a separately verified free-energy evaluator and test energy behavior without assuming it from mean conservation.
